In [ ]:
!pip install librosa

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 310.6 kB/s  0:00:11315.1 kB/s eta 0:00:01
   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/56.3 MB 115.7 kB/s eta 0:07:19

In [1]:
import gc
import librosa
import numpy as np
import torch


# ==================================
# TREE INFERENCE
# ==================================

def classify_tree_recording(signal, sr, model_path="tree_model_full.pt",
                            threshold=0.5, window_seconds=10):


    # ==========================
    # TEAGER (TRAINED VERSION)
    # ==========================

    def teager_energy(signal):

        signal = np.asarray(signal)

        out = np.zeros_like(signal)

        out[1:-1] = (
            signal[1:-1]**2
            -
            signal[:-2] * signal[2:]
        )

        out = np.maximum(out, 0)

        return np.sqrt(out + 1e-10)


    # ==========================
    # MFCC
    # ==========================

    def extract_mfcc(signal):

        mfcc = librosa.feature.mfcc(
            y=signal,
            sr=8000,
            n_mfcc=13,
            n_fft=1024,
            hop_length=256
        )

        return mfcc.astype(
            np.float32
        )


    # ==========================
    # VALIDATE INPUT
    # ==========================

    if sr != 8000:

        return {
            "label": "RETAKE",
            "message": "Expected 8000 Hz sample rate"
        }


    duration = len(signal) / 8000


    if duration < window_seconds:

        return {
            "label": "RETAKE",
            "message": "Recording shorter than 10 seconds"
        }


    # ==========================
    # DEVICE
    # ==========================

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    # ==========================
    # LOAD MODEL
    # ==========================
    model = torch.jit.load(
        "tree_model.pth",
        map_location=device
    )

    model.eval()
    try:

        window_size = (
            window_seconds
            *
            8000
        )


        windows = []

        full = len(signal) // window_size

        remainder = len(signal) % window_size


        for i in range(full):

            start = i * window_size

            windows.append(
                signal[
                    start:
                    start + window_size
                ]
            )


        if remainder > 0:

            windows.append(
                signal[
                    -window_size:
                ]
            )


        windows = windows[:4]


        probs = []

        labels = []


        # ==========================
        # INFERENCE
        # ==========================

        with torch.no_grad():

            for w in windows:

                raw = extract_mfcc(
                    w
                )

                tea = extract_mfcc(
                    teager_energy(w)
                )


                x = np.stack(
                    [
                        raw,
                        tea
                    ],
                    axis=0
                )


                x = (
                    torch.tensor(
                        x,
                        dtype=torch.float32
                    )
                    .unsqueeze(0)
                    .to(device)
                )


                prob = (
                    torch.sigmoid(
                        model(x)
                    )
                    .item()
                )


                pred = int(
                    prob >= threshold
                )


                probs.append(
                    prob
                )

                labels.append(
                    pred
                )


        # ==========================
        # FINAL DECISION
        # ==========================

        n = len(labels)

        positives = sum(
            labels
        )


        if positives == 0:

            final = "CLEAN"


        elif n <= 2:

            final = "INFESTED"


        elif positives == 1:

            final = "SUSPICIOUS"


        elif positives >= (n / 2):

            final = "INFESTED"


        else:

            final = "SUSPICIOUS"


        return {

            "label":
                final,

            "windows_used":
                n,

            "positive_windows":
                positives,

            "window_probs":
                probs,

            "window_labels":
                labels,

            "duration_sec":
                round(
                    duration,
                    2
                )

        }


    finally:

        del model

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

        gc.collect()

## e

In [2]:
audio, sr = librosa.load(
    "clean_4.wav",
    sr=None
)

result = classify_tree_recording(
    audio,
    sr
)

print(result)

{'label': 'CLEAN', 'windows_used': 1, 'positive_windows': 0, 'window_probs': [0.00026736731524579227], 'window_labels': [0], 'duration_sec': 10.0}
